# 02 Crazyflie — Changing UWB Measurement Noise: Evaluation

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from core import SimulationResult
from evaluation import Evaluator
from evaluation.aligners import NearestTimeIndexAligner
from evaluation.core import EvaluationResult
from evaluation.metrics import PositionRMSE, NEES
from evaluation.metrics.attitude_rmse import AttitudeRMSE
from evaluation.metrics.velocity_rmse import VelocityRMSE
from evaluation.runners import RustRunnerCrazyflie
from evaluation.visualizer import EvaluationVisualizer

SIMULATION_NAME = "02_crazyflie_changing_measurement_noise"
SIM_DATA_PATH = Path("../../simulated_data/rotorpy/noise/") / SIMULATION_NAME
EVAL_DATA_PATH = Path("../../evaluation_data/rotorpy/noise/") / SIMULATION_NAME
EVAL_DATA_PATH.mkdir(parents=True, exist_ok=True)

## Data Loading

In [ ]:
simulation_result = SimulationResult.load(str(SIM_DATA_PATH / (SIMULATION_NAME + ".pkl")))

t_total = simulation_result.ground_truth.time[-1]
t1 = t_total / 3.0
t2 = 2.0 * t_total / 3.0

PHASES = {
    "Normal (x1.0)": (0.0, t1),
    "High   (x2.0)": (t1, t2),
    "Low    (x0.5)": (t2, t_total),
}

print(f"Simulation: {SIMULATION_NAME}")
print(f"Duration: {t_total:.1f} s  |  Phase boundaries: {t1:.1f} s, {t2:.1f} s")

## Evaluation

In [ ]:
VARIANTS = {
    # "no_tof": dict(do_correction_tof=False, adaptive_uwb=None, adaptive_tof=None),
    "tof": dict(do_correction_tof=True, adaptive_uwb=None, adaptive_tof=None),
    "tof_sw": dict(do_correction_tof=True, adaptive_uwb="sliding_window", adaptive_tof="sliding_window"),
    "tof_ema": dict(do_correction_tof=True, adaptive_uwb="ema", adaptive_tof="ema"),
}

aligner = NearestTimeIndexAligner()
metrics = [PositionRMSE(), VelocityRMSE(), AttitudeRMSE(), NEES()]

results: dict[str, EvaluationResult] = {}
for variant_name, runner_kwargs in VARIANTS.items():
    runner = RustRunnerCrazyflie(use_noisy=True, do_correction_uwb=True, **runner_kwargs)
    evaluator = Evaluator(runner, metrics, aligner)
    ev = evaluator.evaluate(simulation_result)
    results[variant_name] = ev
    # ev.save(str(EVAL_DATA_PATH / f"{variant_name}.pkl"))
    print(f"{variant_name:10s}  pos={ev.metrics['position_rmse'].value:.4f} m  "
          f"vel={ev.metrics['velocity_rmse'].value:.4f} m/s  "
          f"att={ev.metrics['attitude_rmse'].value:.4f} deg")

## Accuracy

In [ ]:
METRICS = ["position_rmse", "velocity_rmse", "attitude_rmse"]

rows = []
for m in METRICS:
    row = {"Metric": m, "Unit": results["tof"].metrics[m].unit}
    for v, ev in results.items():
        row[v] = f"{ev.metrics[m].value:.4f}"
    rows.append(row)

pd.DataFrame(rows).set_index("Metric")

## Per-Phase Analysis

In [ ]:
from scipy.spatial.transform import Rotation


def _phase_rmse(ev: EvaluationResult, t_start: float, t_end: float) -> dict:
    t = ev.estimate.time
    mask = (t >= t_start) & (t <= t_end)
    if not np.any(mask):
        return {}
    pos_err = ev.estimate.position[mask] - ev.ground_truth.position[mask]
    vel_err = ev.estimate.velocity[mask] - ev.ground_truth.velocity[mask]
    r_err = (
            Rotation.from_quat(ev.estimate.attitude[mask], scalar_first=True)
            * Rotation.from_quat(ev.ground_truth.attitude[mask], scalar_first=True).inv()
    )
    att_err = r_err.as_rotvec()
    return {
        "pos [m]": float(np.sqrt(np.mean(np.sum(pos_err ** 2, axis=1)))),
        "vel [m/s]": float(np.sqrt(np.mean(np.sum(vel_err ** 2, axis=1)))),
        "att [deg]": float(np.degrees(np.sqrt(np.mean(np.sum(att_err ** 2, axis=1))))),
    }


for variant_name, ev in results.items():
    rows = []
    for phase_label, (t_s, t_e) in PHASES.items():
        r = _phase_rmse(ev, t_s, t_e)
        rows.append({"Phase": phase_label, **{k: f"{v:.4f}" for k, v in r.items()}})
    print(f"\n{variant_name}")
    display(pd.DataFrame(rows).set_index("Phase"))

## Trajectory

In [ ]:
PLOT_VARIANT = "tof"  # "no_tof" | "tof" | "tof_sw" | "tof_ema"

fig = EvaluationVisualizer.plot_multiple_flight_trajectories(
    [results[PLOT_VARIANT]],
    labels=[PLOT_VARIANT],
    show_estimate=True,
    title=f"Trajectory — {SIMULATION_NAME} ({PLOT_VARIANT})",
)
fig.update_layout(font=dict(size=14), width=900, height=700)
fig.show()

## State Errors

In [ ]:
import matplotlib.pyplot as plt

PLOT_VARIANT = "tof"  # "no_tof" | "tof" | "tof_sw" | "tof_ema"

fig = EvaluationVisualizer(results[PLOT_VARIANT]).plot_state_errors(show_events=False)
for ax in fig.axes:
    ax.axvline(t1, color='darkorange', linestyle='--', linewidth=1.2, label='phase boundary')
    ax.axvline(t2, color='darkorange', linestyle='--', linewidth=1.2)
fig.axes[0].legend(loc='upper right', fontsize=8)
fig.suptitle(f"State Errors — {PLOT_VARIANT}", fontsize=11)
fig.subplots_adjust(top=0.93)
plt.show()

## Consistency (NEES)

In [ ]:
PLOT_VARIANT = "tof"  # "no_tof" | "tof" | "tof_sw" | "tof_ema"

ax = EvaluationVisualizer(results[PLOT_VARIANT]).plot_nees_groups_over_time()
ax.axvline(t1, color='darkorange', linestyle='--', linewidth=1.2, label='phase boundary')
ax.axvline(t2, color='darkorange', linestyle='--', linewidth=1.2)
ax.legend(loc='upper right')
ax.set_title(f"NEES — {PLOT_VARIANT}")
ax.figure.tight_layout()
plt.show()

## Adaptive Measurement Variances

In [ ]:
fig = EvaluationVisualizer(results["tof_sw"]).plot_measurement_variances(as_stddev=True)
for ax in fig.axes:
    ax.axvline(t1, color='darkorange', linestyle='--', linewidth=1.2)
    ax.axvline(t2, color='darkorange', linestyle='--', linewidth=1.2)
fig.suptitle("Estimated measurement std-dev — Sliding Window", fontsize=11)
fig.subplots_adjust(top=0.93)
plt.show()
# fig.savefig("crazyflie-uwb-std-dev-sw.png", dpi=400, bbox_inches="tight", pad_inches=0.3)

In [ ]:
fig = EvaluationVisualizer(results["tof_ema"]).plot_measurement_variances(as_stddev=True)
for ax in fig.axes:
    ax.axvline(t1, color='darkorange', linestyle='--', linewidth=1.2)
    ax.axvline(t2, color='darkorange', linestyle='--', linewidth=1.2)
fig.suptitle("Estimated measurement std-dev — EMA", fontsize=11)
fig.subplots_adjust(top=0.93)
plt.show()
# fig.savefig("crazyflie-uwb-std-dev-ema.png", dpi=400, bbox_inches="tight", pad_inches=0.3)


## Variant Comparison

In [ ]:
fig = EvaluationVisualizer.plot_state_error_norms_comparison(
    [results["tof"], results["tof_sw"], results["tof_ema"]],
    labels=["Static (bias compensated)", "SW (bias compensated)", "EMA (bias compensated)"],
)
plt.show()
# fig.savefig("crazyflie_adaptive_position_error_changing_variance.png", dpi=600, bbox_inches="tight", pad_inches=0.3)
